<h1>Introduction Summary</h1>
You are building an NLP chatbot capable of slot-filling for natural language understanding (NLU) tasks. The chatbot uses a deep learning model based on BERT for token classification. A BIO-tagged dataset is used to train the model to identify and extract information (slots) from user input. Key steps include preprocessing data, tokenizing inputs, aligning labels, creating datasets, and fine-tuning a BERT model for sequence tagging. The trained model can predict slot information in new sentences for chatbot functionalities.

Some preproccessing function:

In [1]:
from nltk.corpus import stopwords
import re
import nltk

# Download stopwords if not already available
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def custom_preprocess(sentence):
    tokens = sentence.split()  # Basic tokenization
    processed_tokens = []
    for token in tokens:
        if token.lower() in stop_words:  # Remove stopwords
            continue
        elif token.lower() == "km":  # Replace "km" with <DIS>
            processed_tokens.append("<DIS>")
        elif re.match(r"^\d+(\.\d+)?$", token):  # Replace numbers with <DIG>
            processed_tokens.append("<DIG>")
        else:
            processed_tokens.append(token)
    return processed_tokens


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


<h1>1. Ensure the Dataset is Clean</h1>
Ensure your dataset (crafted_dataset_1k.csv) has:

* sentence: Input text (full sentences).
* bio_tags: Corresponding BIO tags aligned with tokens.
* tokens: Tokens for each sentence, comma-separated.
Preview a few rows to confirm the structure:

In [3]:
import pandas as pd

# Load dataset
file_path = "crafted_dataset_30k.csv"
data = pd.read_csv(file_path)

# Preview data
print(data.head())


                                            sentence  \
0     {difficulty: challenging} running track please   
1  Looking for a route with coffee shops along th...   
2  {start_location: weizmann} {loca_start_num: 61...   
3  Looking for a nice {difficulty: very hard} pat...   
4  from {start_location: sephardim street} {loca_...   

                                              tokens  \
0                challenging, running, track, please   
1  Looking, for, a, route, with, coffee, shops, a...   
2  weizmann, 61, to, habustan, street, 10, ,, 15....   
3  Looking, for, a, nice, very, hard, path, from,...   
4  from, sephardim, street, 91, need, slightly, c...   

                                            bio_tags  
0                              B-difficulty, O, O, O  
1  O, O, O, O, O, O, O, O, O, O, O, B-route_lengt...  
2  B-start_location, B-loca_start_num, O, B-end_l...  
3  O, O, O, O, B-difficulty, I-difficulty, O, O, ...  
4  O, B-start_location, I-start_location, B-loca_..

<h1>2. Split the Dataset</h1>
Split the dataset into training, validation, and testing sets:

In [4]:
from sklearn.model_selection import train_test_split

# Split into train, validation, and test sets
train_data, temp_data = train_test_split(data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)


<h1>3. Preprocess Data</h1>
Extract tokens and tags, converting them into lists of lists:

In [5]:
def preprocess_data(data):
    sentences = data["tokens"].apply(lambda x: custom_preprocess(" ".join(x.split(",")))).tolist()
    tags = data["bio_tags"].apply(lambda x: x.split(",")).tolist()
    return sentences, tags

train_sentences, train_tags = preprocess_data(train_data)
val_sentences, val_tags = preprocess_data(val_data)
test_sentences, test_tags = preprocess_data(test_data)


<h1>4. Define Tag Mapping</h1>
Create mappings for BIO tags:

In [6]:
tag2id = {
    "O": 0, "B-difficulty": 1, "I-difficulty": 2,
    "B-route_length": 3, "I-route_length": 4,
    "B-start_location": 5, "I-start_location": 6,
    "B-start_number": 7, "I-start_number": 8,
    "B-end_location": 9, "I-end_location": 10,
    "B-loca_end_num": 11, "I-loca_end_num": 12,
    "B-loca_start_num": 13, "I-loca_start_num": 14,
}
id2tag = {v: k for k, v in tag2id.items()}

<h1>5. Tokenize and Align Labels</h1>
Ensure tokens and labels are properly aligned using word_ids from the tokenizer:

In [7]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(sentences, tags):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )

    labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = []
        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)  # Ignore special tokens
            elif word_id < len(label):
                aligned_labels.append(tag2id[label[word_id].strip()])
            else:
                aligned_labels.append(-100)  # In case of mismatch
        labels.append(aligned_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize datasets
train_inputs = tokenize_and_align_labels(train_sentences, train_tags)
val_inputs = tokenize_and_align_labels(val_sentences, val_tags)
test_inputs = tokenize_and_align_labels(test_sentences, test_tags)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

<h1>6. Create PyTorch Dataset</h1>
Define a dataset class:

In [8]:
import torch
from torch.utils.data import Dataset

class TokenClassificationDataset(Dataset):
    def __init__(self, inputs):
        self.inputs = inputs

    def __len__(self):
        return len(self.inputs["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.inputs["input_ids"][idx],
            "attention_mask": self.inputs["attention_mask"][idx],
            "labels": self.inputs["labels"][idx],
        }

train_dataset = TokenClassificationDataset(train_inputs)
val_dataset = TokenClassificationDataset(val_inputs)
test_dataset = TokenClassificationDataset(test_inputs)


<h1>7. Initialize and Train the Model</h1>
Load the pre-trained BERT model and set training arguments:

In [9]:
# API_KEY: 611797cd91efcf043036bb77a209ff83138c41e3
from transformers import BertForTokenClassification, TrainingArguments, Trainer

model = BertForTokenClassification.from_pretrained("bert-base-cased", num_labels=len(tag2id))

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    save_steps=500,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

trainer.train()


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-9-c236f70e2027>:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch,Training Loss,Validation Loss
1,0.382000,0.206892
2,0.173800,0.151765
3,0.104100,0.115431


TrainOutput(global_step=3939, training_loss=0.2670531484735106, metrics={'train_runtime': 938.3242, 'train_samples_per_second': 67.141, 'train_steps_per_second': 4.198, 'total_flos': 1832865109110000.0, 'train_loss': 0.2670531484735106, 'epoch': 3.0})

<h1>8. Evaluate the Model</h1>
Use the test dataset and the seqeval library for evaluation:

In [10]:
!pip install seqeval
from seqeval.metrics import classification_report

# Get predictions
predictions, labels, _ = trainer.predict(test_dataset)

# Convert predictions and true labels back to tags
predicted_tags = [[id2tag[id] for id in pred if id != -100] for pred in predictions.argmax(axis=2)]
true_tags = [[id2tag[id] for id in label if id != -100] for label in labels]

true_lengths = [len(seq) for seq in true_tags]
pred_lengths = [len(seq) for seq in predicted_tags]

# Truncate predictions to match the true tag lengths
adjusted_predicted_tags = [
    pred[:len(true)]
    for pred, true in zip(predicted_tags, true_tags)
]

# Verify the lengths are now aligned
true_lengths = [len(seq) for seq in true_tags]
adjusted_pred_lengths = [len(seq) for seq in adjusted_predicted_tags]

print(f"True lengths: {true_lengths}")
print(f"Adjusted predicted lengths: {adjusted_pred_lengths}")



# Evaluate using seqeval
# print(classification_report(true_tags, predicted_tags))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=30a6208829f4bf07c48cb4405890e5e23d91d9935958301e7deaabdaad344167
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval


True lengths: [20, 12, 30, 10, 9, 24, 10, 18, 32, 12, 3, 13, 5, 19, 30, 3, 4, 33, 3, 14, 21, 14, 8, 4, 30, 12, 5, 15, 4, 10, 9, 11, 8, 3, 13, 9, 5, 16, 19, 11, 20, 9, 11, 4, 8, 20, 4, 15, 11, 18, 5, 13, 16, 12, 22, 22, 6, 16, 19, 21, 10, 13, 12, 13, 21, 6, 4, 24, 4, 28, 4, 19, 11, 4, 32, 10, 8, 8, 7, 4, 4, 5, 13, 7, 25, 11, 19, 23, 8, 4, 12, 20, 5, 16, 30, 19, 4, 12, 20, 16, 35, 4, 32, 5, 22, 13, 15, 19, 3, 4, 8, 21, 7, 5, 16, 5, 18, 6, 33, 24, 20, 34, 26, 16, 16, 14, 12, 10, 23, 4, 9, 19, 36, 5, 26, 23, 4, 12, 4, 15, 10, 29, 6, 19, 18, 5, 11, 30, 15, 5, 23, 22, 9, 26, 4, 20, 10, 10, 20, 10, 18, 17, 30, 13, 13, 23, 37, 16, 7, 7, 8, 29, 20, 29, 5, 9, 28, 18, 14, 15, 4, 5, 12, 3, 12, 11, 7, 19, 21, 4, 5, 31, 12, 21, 14, 12, 22, 16, 16, 13, 5, 11, 13, 19, 3, 31, 21, 4, 5, 19, 35, 14, 11, 12, 4, 8, 5, 23, 32, 9, 21, 5, 9, 21, 24, 22, 17, 11, 4, 12, 5, 10, 25, 8, 26, 32, 7, 13, 44, 4, 21, 16, 12, 10, 4, 34, 13, 4, 21, 17, 8, 21, 7, 17, 28, 3, 10, 4, 11, 4, 9, 11, 22, 30, 25, 17, 8, 7, 14, 4

<h1>9. Inference</h1>
To use the model for slot filling on new sentences:

In [19]:
import torch

def predict_slots(sentence):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Preprocess the sentence
    processed_tokens = custom_preprocess(sentence)

    # Tokenize the processed tokens
    inputs = tokenizer(processed_tokens, is_split_into_words=True, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = outputs.logits.argmax(dim=2).squeeze().tolist()
    predicted_tags = [id2tag[p] for p in predictions if p != -100]
    return list(zip(processed_tokens, predicted_tags))


print(predict_slots("give me a 3 km route starting from my location"))


[('give', 'O'), ('<DIG>', 'O'), ('<DIS>', 'O'), ('route', 'O'), ('starting', 'O'), ('location', 'O')]


<h3>This step-by-step process ensures:</h3>
<br>1. Data is clean and correctly tokenized.
<br>2. Tokens and tags are properly aligned.
<br>3. Model training and evaluation are seamless.fine-tune if necessary.